In [13]:
# 강남구 데이터 10년치 수집함
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
from dotenv import load_dotenv
import os
import glob

In [14]:
load_dotenv()
api_key = os.getenv("KAKAO_API_KEY")

In [15]:

# 사용할 컬럼만 지정 (NO 컬럼 제외)
columns_to_use = [
    '시군구', '단지명', '전용면적(㎡)', '계약년월', '계약일',
    '거래금액(만원)', '동', '층', '건축년도', '도로명'
]

# 병합할 CSV 파일이 들어있는 폴더 경로
folder_path = "../../data/raw/apt_sale"
gangnam_path = "../../data/raw/apt_sale/gangnam"
# 해당 경로의 모든 .csv 파일 리스트 얻기
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
# 병합할 데이터프레임 저장할 리스트
df_list = []

# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
        print(f"읽기 완료: {os.path.basename(file)}")
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")
# --- 여기까지 전국
# 아래부터 강남만
csv_files = glob.glob(os.path.join(gangnam_path, "*.csv"))
# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
        print(f"읽기 완료: {os.path.basename(file)}")
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")




# 데이터프레임 병합
df = pd.concat(df_list, ignore_index=True)

# 병합된 결과 출력
print(f"\n 병합 완료: 총 {len(df)}건")

읽기 완료: 아파트(매매)_실거래가_20250915192726.csv
읽기 완료: 아파트(매매)_실거래가_20250915192733.csv
읽기 완료: 아파트(매매)_실거래가_20250915192719.csv
읽기 완료: 아파트(매매)_실거래가_20250915192730.csv
읽기 완료: 아파트(매매)_실거래가_20250915192709.csv
읽기 완료: 아파트(매매)_실거래가_20250915192735.csv
읽기 완료: 아파트(매매)_실거래가_20250915192723.csv
읽기 완료: 아파트(매매)_실거래가_20250915192750.csv
읽기 완료: 아파트(매매)_실거래가_20250915192747.csv
읽기 완료: 아파트(매매)_실거래가_20250915192743.csv
읽기 완료: 아파트(매매)_실거래가_20250915192759.csv
읽기 완료: 아파트(매매)_실거래가_20250915192808.csv
읽기 완료: 아파트(매매)_실거래가_20250915192738.csv
읽기 완료: 아파트(매매)_실거래가_20250915192714.csv
읽기 완료: 아파트(매매)_실거래가_20250915192703.csv

 병합 완료: 총 64057건


In [16]:
min_contract = df['계약년월'].min()
max_contract = df['계약년월'].max()
print(f"데이터는 {min_contract} ~ {max_contract}까지의 거래로 이루어져 있습니다.")

데이터는 201007 ~ 202506까지의 거래로 이루어져 있습니다.


# 면적당 단가 계산

In [17]:
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)
df['면적당 단가(만원)'] = df['거래금액(만원)'] / df['전용면적(㎡)']

In [18]:
df.head()

,시군구,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,건축년도,도로명,면적당 단가(만원)
0,서울특별시 강남구 대치동,래미안대치팰리스,84.990,201606,30,140000,-,3,2015,삼성로51길 37,1647.252618
1,서울특별시 강남구 대치동,대치아이파크,59.960,201606,30,97000,-,13,2008,선릉로 222,1617.745163
2,서울특별시 강남구 삼성동,삼성동힐스테이트 1단지,31.402,201606,30,64500,-,12,2008,학동로68길 29,2054.009299
3,서울특별시 강남구 수서동,까치마을,49.500,201606,30,62500,-,4,1993,광평로19길 10,1262.626263
4,서울특별시 강남구 논현동,두산위브1단지,84.993,201606,30,81000,-,5,2004,학동로46길 38,953.019660


# 아파트 나이 계산

In [19]:
df['계약년도'] = df['계약년월'].astype(str).str[:4].astype(int)
df['아파트 나이'] = df['계약년도'] - df['건축년도']

# 거래 순으로 나열 및 필요 없는 컬럼 삭제

In [20]:
df['구'] = df['시군구'].str.extract(r'(\S+구)')

# 계약연-월-일을 기준으로 시계열 정렬
df['계약일자'] = df['계약년월'].astype(str) + df['계약일'].astype(str).str.zfill(2)
df['계약일자'] = pd.to_datetime(df['계약일자'], format='%Y%m%d')

df = df.sort_values('계약일자').reset_index(drop=True)
#df.drop(['시군구','계약년월','계약일','동','계약년도'], axis=1, inplace=True)
# 아래는 동이 없는 버전
df.drop(['시군구','계약년월','계약일','계약년도'], axis=1, inplace=True)

In [21]:
df.head()

,단지명,전용면적(㎡),거래금액(만원),동,층,건축년도,도로명,면적당 단가(만원),아파트 나이,구,계약일자
0,한화진넥스빌,39.200,18000,-,15,2001,언주로86길 11,459.183673,9,강남구,2010-07-01
1,도곡스타클래스,111.380,72500,-,14,2007,남부순환로 2615,650.924762,3,강남구,2010-07-01
2,경남아너스빌,81.013,55000,-,3,2002,언주로85길 13,678.903386,8,강남구,2010-07-01
3,까치마을,39.600,33000,-,7,1993,광평로19길 10,833.333333,17,강남구,2010-07-01
4,우정에쉐르멤버스,36.190,20001,-,4,2004,선릉로87길 14,552.666482,6,강남구,2010-07-01


In [22]:
df[df['도로명'] =='언주로 332']

,단지명,전용면적(㎡),거래금액(만원),동,층,건축년도,도로명,면적당 단가(만원),아파트 나이,구,계약일자
104,역삼푸르지오,59.8848,67750,-,5,2006,언주로 332,1131.338837,4,강남구,2010-07-19
151,역삼푸르지오,84.9097,94000,-,14,2006,언주로 332,1107.058440,4,강남구,2010-07-28
260,역삼푸르지오,84.9097,96750,-,10,2006,언주로 332,1139.445788,4,강남구,2010-08-17
298,역삼푸르지오,84.9097,96500,-,10,2006,언주로 332,1136.501483,4,강남구,2010-08-21
315,역삼푸르지오,84.9097,93000,-,5,2006,언주로 332,1095.281222,4,강남구,2010-08-23
...,...,...,...,...,...,...,...,...,...,...,...
63235,역삼푸르지오,59.8848,255000,111,6,2006,언주로 332,4258.175697,19,강남구,2025-05-05
63507,역삼푸르지오,84.9097,314500,103,3,2006,언주로 332,3703.934886,19,강남구,2025-06-02
63860,역삼푸르지오,84.9097,320000,104,7,2006,언주로 332,3768.709582,19,강남구,2025-06-25
64032,역삼푸르지오,84.9097,314000,-,3,2006,언주로 332,3698.046277,19,강남구,2025-06-30


In [30]:
df[df['동'] != '-']

,단지명,전용면적(㎡),거래금액(만원),동,층,건축년도,도로명,면적당 단가(만원),아파트 나이,구,계약일자
55369,목련타운,99.7900,175000,109,6,1993,광평로19길 15,1753.682734,30,강남구,2023-01-01
55371,성원대치2단지아파트,33.1800,81700,215,2,1992,개포로109길 9,2462.326703,31,강남구,2023-01-02
55372,경남1,166.4800,290000,5,8,1984,언주로 110,1741.950985,39,강남구,2023-01-02
55373,래미안블레스티지,49.9090,129000,218,10,2019,선릉로 8,2584.704162,4,강남구,2023-01-02
55374,개포주공5단지,74.2500,210000,503,10,1983,삼성로4길 17,2828.282828,40,강남구,2023-01-03
...,...,...,...,...,...,...,...,...,...,...,...
64042,삼성동대성유니드,83.9300,165000,101,2,2004,테헤란로77길 26,1965.923984,21,강남구,2025-06-30
64044,강남자곡아이파크,59.9600,145000,704,5,2014,자곡로 175,2418.278853,11,강남구,2025-06-30
64046,역삼I'PARK,28.2460,73000,201,2,2006,역삼로 307,2584.436734,19,강남구,2025-06-30
64048,래미안대치팰리스,91.9300,450000,202,7,2015,삼성로51길 35,4895.028826,10,강남구,2025-06-30


In [28]:
df['단지명'].nunique()

690

In [29]:
df['단지명'].unique()

array(['한화진넥스빌', '도곡스타클래스', '경남아너스빌', '까치마을', '우정에쉐르멤버스', '삼익대청아파트',
       '개포주공6단지', '수서한아름', '삼성', '힐스빌7', '매봉삼성', '필로스(193-45)',
       "역삼I'PARK", '청담대림이-편한세상', '은마', '푸른마을아파트101동~111동', '주공2',
       '마일스디오빌', '개포주공1단지', '동산', '도곡렉슬', '대우디오빌', '삼성래미안',
       '우찬현대(1260-4)', '테헤란로대우아이빌(891-6)', '개나리푸르지오', '미켈란107',
       '현대5차(71,72동)', '청실1', '개포주공7단지', 'SK허브프리모', 'e-편한세상', '개포주공4단지',
       '한보미도맨션1', '동부센트레빌', '개포우성1', '한보미도맨션2', '삼부아파트102동', '한양2',
       '래미안펜타빌', '역삼래미안', '미성2차', '성원대치2단지아파트', '청담동삼성1차(101동)', '역삼예명',
       '개포주공3단지', '월드메르디앙102동', '쌍용', '샘터마을', '대치동우정에쉐르1', '우정에쉐르',
       '아름빌(889-74)', '호정빌라트', '신동아', '아델하우스', '대치현대', '한신엠비씨', '목련타운',
       '타워팰리스2', '신동아(22)', '대치아이파크', '현대2차(10,11,20,23,24,25동)', '한솔마을',
       '금호어울림', '개포자이', '석탑아파트101동', '역삼푸르지오', 'LG선릉에클라트(A)', '대치삼성',
       '삼익', '청담현대3차아파트', '수서', '우성8', '대우디오빌플러스', '현대비젼21', '포스코더샵',
       '삼성동한솔아파트', '개포우성2', '채널리저브', '리더스빌', '래미안삼성2차', '롯데캐슬프레미어',
       '역삼동프라임', '대치한신휴플러스', '개포주공5단지', '

In [34]:
df = df[df['구'] == '강남구']

In [35]:
# 이상치 제거 함수 예시 (IQR 방식 등 사용자 정의 필요)
def remove_price_outliers(group):
    q1 = group['거래금액(만원)'].quantile(0.25)
    q3 = group['거래금액(만원)'].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    filtered = group[(group['거래금액(만원)'] >= lower) & (group['거래금액(만원)'] <= upper)]
    return filtered

def calculate_alpha_from_age_count(age, count, N=30):
 
    raw_alpha = (1 - age / N) * np.log2(count + 1)
    alpha = max(0, min(1, raw_alpha))
    return alpha

def representative_price(prices, dates, age, N=30):

    if len(prices) == 0:
        return None  # 거래 없음 → 대표값 계산 불가

    # 가중치 alpha 계산
    count = len(prices)
    alpha = calculate_alpha_from_age_count(age, count, N)

    # 평균 거래가 (P̄)
    avg_price = np.mean(prices)

    # 최신 거래가 (P_latest)
    latest_index = np.argmax(dates)  # 거래일 기준 최대값 인덱스
    latest_price = prices[latest_index]

    # 대표 거래가 계산
    rep_price = alpha * avg_price + (1 - alpha) * latest_price
    return rep_price


def calculate_alpha_row(group, N=30):
    age = group['아파트 나이'].iloc[0]  # 해당 그룹의 아파트 나이
    count = len(group)  # 그룹 내 거래 수

    alpha = calculate_alpha_from_age_count(age, count, N)

    # 대표 row는 그룹의 첫 row 기준으로 생성
    row = group.iloc[0].copy()
    row['alpha'] = alpha
    return pd.DataFrame([row])

In [36]:
# 1. 월 단위 컬럼 생성 (이미 있다면 생략 가능)
# '계약일자' 컬럼에서 'YYYYMM' 형식의 계약년월을 다시 만듭니다.
df['계약년월'] = df['계약일자'].dt.strftime('%Y%m')

# 2. 이상치 제거 (월별 그룹 기준)
# groupby에 '계약년월'을 추가합니다.
df_filtered = df.groupby(['도로명', '단지명', '전용면적(㎡)', '계약년월'], group_keys=False)\
                .apply(remove_price_outliers)\
                .reset_index(drop=True)

# 3. 대표 row 추출 (월별 그룹 기준)
# groupby에 '계약년월'을 추가합니다.
df_monthly_representative = df_filtered.groupby(['도로명', '단지명', '전용면적(㎡)', '계약년월'], group_keys=False)\
                                     .apply(calculate_alpha_row)\
                                     .reset_index(drop=True)

# 4. 최종 정렬
df_final = df_monthly_representative.sort_values('계약일자').reset_index(drop=True)

print(f"월별 처리 후 데이터 개수: {len(df_final)}")

/var/folders/sq/2pgmkw912zj1zpn9tz6pjdmr0000gn/T/ipykernel_81942/3088890929.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(remove_price_outliers)\


월별 처리 후 데이터 개수: 41070


/var/folders/sq/2pgmkw912zj1zpn9tz6pjdmr0000gn/T/ipykernel_81942/3088890929.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_alpha_row)\


In [37]:
df = df_final

In [38]:
df.drop('거래금액(만원)', axis=1, inplace=True)
df = df.sort_values('계약일자').reset_index(drop=True)
df.head(1)
#df.tail(1)

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha
0,수서한아름,129.45,10,1993,광평로51길 22,672.07416,17,강남구,2010-07-01,201007,0.433333


# 좌표변환

In [39]:
from pathlib import Path

OUTPUT_PATH = Path('../../data/interim/apt/gang_nam_apt_with_long_lat.csv')  

# === 좌표 변환 === #
headers = {'Authorization': 'KakaoAK 531d049fba15b8acbb290989f6988d89'}
#Authorization: KakaoAK ${REST_API_KEY}"
def get_coords(address):
    res = requests.get(
        "https://dapi.kakao.com/v2/local/search/address.json",
        headers=headers,
        params={'query': address}
    )
    if res.status_code == 200 and res.json()['documents']:
        doc = res.json()['documents'][0]
        return doc['x'], doc['y']
    return None, None

longitudes, latitudes = [], []
for address in tqdm(df['도로명'], desc="좌표 변환 중"):
    x, y = get_coords(address)
    longitudes.append(x)
    latitudes.append(y)

df['경도'] = longitudes
df['위도'] = latitudes

# === 최종 정제 및 저장 === #

df['면적당 단가(만원)'] = np.log(df['면적당 단가(만원)'])

# === 수집 못한 위도 경도는 삭제 === #
df.dropna(inplace=True)

# 디렉토리 없으면 생성
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ 저장 완료: {OUTPUT_PATH}")

좌표 변환 중: 100%|█████████████████████████████████| 41070/41070 [51:20<00:00, 13.33it/s]


✅ 저장 완료: ../data/interim/apt/gang_nam_apt_with_long_lat.csv


In [40]:
len(df)

39411

In [48]:
df = df[df['계약일자'] >= '2018-07-01']

In [50]:
len(df)

14575

In [49]:
df.to_csv(OUTPUT_PATH, index=False)

In [51]:
OUTPUT_PATH

PosixPath('../data/interim/apt/gang_nam_apt_with_long_lat.csv')